# Following uninsured-care reimbursements from source to summary

Does the distribution of program dollars and service mix vary across states? This analysis follows the CDC/HRSA export from provider records through cleaning, state totals, and provider concentration.

The rows describe reimbursement activity, not patient counts, utilization rates, or health outcomes. That distinction guides how the results are grouped and interpreted.

## Source and preparation

- **Load:** Read `../data/claims_reimbursement.csv` into DuckDB; refresh it with `Rscript R/download_cdc_datasets.R`.
- **Grain:** One row per provider/city/state. Aggregation happens only after checking the source fields.
- **Clean:** Trim text fields, standardize state abbreviations, and cast payment amounts to decimals.
- **Audit:** Check for nulls, negative amounts, and rows that are zero across all payment categories before applying any zero-fill rule.
- **Analyze:** Build state service-mix summaries and rank provider/state totals with window functions.

The notebook runs SQL through a Python DuckDB connection, keeping the import and transformations together. Missing or invalid amounts are zero-filled only in the summary view, after their counts are visible.

In [1]:
# Load the CDC export into DuckDB and retain its published field names.
import duckdb
from pathlib import Path

con = duckdb.connect()
source_path = Path('../data/claims_reimbursement.csv')
if not source_path.exists():
    source_path = Path('data/claims_reimbursement.csv')
if not source_path.is_file():
    raise FileNotFoundError('Run Rscript R/download_cdc_datasets.R to download the claims source.')

con.execute(f"""
CREATE OR REPLACE TABLE uninsured_claims AS
SELECT *
FROM read_csv_auto('{source_path.as_posix()}',
    header = true,
    normalize_names = false)
""")

# Capture the row count before filtering or aggregation.
source_rows = con.sql('SELECT COUNT(*) AS source_rows FROM uninsured_claims').df()
source_rows

,source_rows
0,50244


In [2]:
# Audit payment fields before applying any imputation.
con.execute('''
CREATE OR REPLACE VIEW claims_payment_quality AS
SELECT
    COUNT(*) AS source_rows,
    COUNT(*) FILTER (WHERE "Claims Paid for Testing" IS NULL) AS testing_missing,
    COUNT(*) FILTER (WHERE "Claims Paid for Treatment" IS NULL) AS treatment_missing,
    COUNT(*) FILTER (WHERE "Claims Paid for Vaccine" IS NULL) AS vaccine_missing,
    COUNT(*) FILTER (WHERE "Claims Paid for Testing" < 0) AS testing_negative,
    COUNT(*) FILTER (WHERE "Claims Paid for Treatment" < 0) AS treatment_negative,
    COUNT(*) FILTER (WHERE "Claims Paid for Vaccine" < 0) AS vaccine_negative,
    COUNT(*) FILTER (WHERE "Claims Paid for Testing" = 0
        AND "Claims Paid for Treatment" = 0
        AND "Claims Paid for Vaccine" = 0) AS all_zero_rows
FROM uninsured_claims
''')
display(con.sql('SELECT * FROM claims_payment_quality').df())

# Standardize provider/location labels and cast payments to decimals. The cleaned
# view zero-fills invalid or missing amounts only after their counts are visible.
con.execute('''
CREATE OR REPLACE VIEW claims_clean AS
SELECT
    NULLIF(TRIM("Provider Name"), '') AS provider_name,
    UPPER(TRIM(State)) AS state,
    TRIM(City) AS city,
    COALESCE(TRY_CAST("Claims Paid for Testing" AS DECIMAL(14, 2)), 0) AS testing_paid,
    COALESCE(TRY_CAST("Claims Paid for Treatment" AS DECIMAL(14, 2)), 0) AS treatment_paid,
    COALESCE(TRY_CAST("Claims Paid for Vaccine" AS DECIMAL(14, 2)), 0) AS vaccine_paid
FROM uninsured_claims
''')
con.sql('SELECT * FROM claims_clean LIMIT 10').df()

,source_rows,testing_missing,treatment_missing,vaccine_missing,testing_negative,treatment_negative,vaccine_negative,all_zero_rows
0,50244,0,0,0,0,0,0,0


,provider_name,state,city,testing_paid,treatment_paid,vaccine_paid
0,ARCARE,AR,KENSENT,40.0,579.0,360.0
1,CITY OF PAGE,AZ,PAGE,0.0,1060.0,0.0
2,HOMETOWN NURSING,CA,BERKELEY,200.0,150.0,0.0
3,IMMUNO ONCOLOGY CLINIC,CA,IRVINE,1560.0,0.0,12640.0
4,HCAHEALTHONE LLC,CO,ENGLEWOOD,0.0,13388.0,0.0
5,ALACHUA PHARMACY INC,FL,ALACHUA,0.0,0.0,19137.0
6,ST CLOUD HMA PHYSICIAN,FL,SAINT CLOUD,23.0,77.0,0.0
7,AGAPEMD LLC,FL,TAVERNIER,0.0,0.0,40.0
8,CASWELL COUNTY HEALTH,GA,PROVIDENCE,0.0,6589.0,0.0
9,HUMAYUNS SC,IL,CHICAGO,0.0,6331.0,0.0


In [3]:
# Aggregate provider rows by state to compare reimbursement and service mix.
con.execute('''
CREATE OR REPLACE VIEW state_reimbursement_summary AS
SELECT
    state,
    COUNT(*) AS provider_rows,
    SUM(testing_paid) AS testing_paid,
    SUM(treatment_paid) AS treatment_paid,
    SUM(vaccine_paid) AS vaccine_paid,
    SUM(testing_paid + treatment_paid + vaccine_paid) AS total_reimbursement,
    ROUND(100.0 * SUM(vaccine_paid) / NULLIF(SUM(testing_paid + treatment_paid + vaccine_paid), 0), 2) AS vaccine_share_pct
FROM claims_clean
GROUP BY state
''')

# Rank states while keeping the underlying category amounts visible.
con.sql('''
SELECT *, DENSE_RANK() OVER (ORDER BY total_reimbursement DESC) AS reimbursement_rank
FROM state_reimbursement_summary
ORDER BY reimbursement_rank
''').df()

,state,provider_rows,testing_paid,treatment_paid,vaccine_paid,total_reimbursement,vaccine_share_pct,reimbursement_rank
0,TX,5295,1.252954e+09,1.803074e+09,99952245.0,3.155980e+09,3.17,1
1,CA,3515,1.923848e+09,3.674612e+08,184760086.0,2.476069e+09,7.46,2
2,IL,2105,1.382318e+09,1.257993e+08,320729636.0,1.828847e+09,17.54,3
3,FL,3336,5.842444e+08,5.722285e+08,106949117.0,1.263422e+09,8.47,4
4,NJ,2172,6.638960e+08,2.926359e+08,132849079.0,1.089381e+09,12.19,5
5,NC,1981,7.762596e+08,2.711267e+08,20945797.0,1.068332e+09,1.96,6
6,NY,2917,8.580092e+08,1.428309e+08,56914703.0,1.057755e+09,5.38,7
7,GA,1978,3.935408e+08,4.074975e+08,41517970.0,8.425564e+08,4.93,8
8,DC,124,6.375463e+08,7.916205e+07,3578179.0,7.202865e+08,0.50,9
9,TN,1279,3.376844e+08,1.724211e+08,8745240.0,5.188507e+08,1.69,10


In [4]:
# Group by provider and state, then calculate each pair's share and cumulative share.
con.sql('''
WITH provider_totals AS (
    SELECT provider_name, state,
           SUM(testing_paid + treatment_paid + vaccine_paid) AS total_reimbursement
    FROM claims_clean
    WHERE provider_name IS NOT NULL
    GROUP BY provider_name, state
), ranked AS (
    SELECT *,
           SUM(total_reimbursement) OVER () AS all_provider_reimbursement,
           SUM(total_reimbursement) OVER (ORDER BY total_reimbursement DESC
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_reimbursement
    FROM provider_totals
)
SELECT provider_name, state, total_reimbursement,
       ROUND(100.0 * total_reimbursement / NULLIF(all_provider_reimbursement, 0), 2) AS provider_share_pct,
       ROUND(100.0 * cumulative_reimbursement / NULLIF(all_provider_reimbursement, 0), 2) AS cumulative_share_pct
FROM ranked
ORDER BY total_reimbursement DESC
LIMIT 25
''').df()

,provider_name,state,total_reimbursement,provider_share_pct,cumulative_share_pct
0,CURATIVE LABS INC,DC,646137850.0,3.44,3.44
1,FULGENT THERAPEUTICS L,CA,507386938.0,2.70,6.14
2,LABORATORY CORPORATION,NC,484273145.0,2.58,8.72
3,WALGREEN CO,IL,258705264.0,1.38,10.10
4,AEGIS SCIENCES CORPORA,TN,230669547.0,1.23,11.32
5,HELIX OPCO LLC,CA,227691475.0,1.21,12.54
6,QUEST DIAGNOSTICS INCO,MO,226616685.0,1.21,13.74
7,NOMI HEALTH INC,UT,226174599.0,1.20,14.95
8,HARRIS HEALTH SYSTEM,TX,198484573.0,1.06,16.00
9,O HARE CLINICAL LAB SE,IL,187153096.0,1.00,17.00


In [5]:
# Show jurisdictions with provider rows that are zero in every payment category.
# An empty result means none appear in the current source snapshot.
con.sql('''
SELECT state, COUNT(*) AS zero_value_rows
FROM claims_clean
WHERE testing_paid = 0 AND treatment_paid = 0 AND vaccine_paid = 0
GROUP BY state
ORDER BY zero_value_rows DESC
''').df()

,state,zero_value_rows


## What the current snapshot shows

The extract contains 50,244 provider/city/state rows. In this download, payment fields have no null or negative values, and no row is zero across all three categories. These checks stay in the workflow because a future refresh may look different.

Texas leads state totals at about $3.16 billion, followed by California ($2.48 billion) and Illinois ($1.83 billion). The mix is different in each: Texas is led by treatment ($1.80 billion), while California is led by testing ($1.92 billion). At provider/state grain, Curative Labs Inc. in DC is the largest pair at about $646 million. The 25 largest named-provider pairs account for 26.44% of named-provider reimbursements in this extract.

That concentration helps locate where reported payments accumulated; it does not tell us how many people received care, what need looked like, or why payments differed. These values describe one downloaded snapshot and should be refreshed before reuse.